# Why `0.1 + 0.2 != 0.3`: how Python stores decimal numbers, and how to compare them safely

<!-- la:orientation -->
Type `0.1 + 0.2 == 0.3` into Python and it answers `False`. This notebook explains exactly why, by looking at how a computer stores a number like `0.1`, and then shows how to compare such numbers safely in your own code.

Why it matters: the same effect turns up in ordinary programs. Add a 0.10 item to a running total ten times and the total is not quite `1.0`, so a check like `if total == 1.0:` silently fails. Once you know where the error comes from, you can predict when it will bite and choose the right fix.

**Who this is for:** you can write basic Python (variables, loops, functions, `print`, f-strings), but you have never learned how numbers are stored inside a computer. No binary is assumed; school fractions are enough. You need Python 3.9 or newer.

**How to use this notebook:** run the cells from top to bottom. Before each experiment there is a **Predict** cell: write your guess there *before* running the code. A wrong guess is useful, because it shows which part of your picture needs fixing.

<!-- la:outcomes -->
By the end you will be able to:

1. Decide whether a decimal number such as `0.375` or `0.1` can be stored exactly in a Python `float`, and justify the answer by writing it as a binary fraction.
2. Reveal the exact value a `float` stores, and use it to explain step by step why a calculation such as `0.1 + 0.2` does not equal `0.3`.
3. Explain why the gap between neighbouring floats grows with the size of the number, and measure that gap with `math.ulp`.
4. Compare floats safely with `math.isclose`, choosing a relative and an absolute tolerance that fit the problem (including comparisons with zero).
5. Decide when to avoid floats entirely and use integers, `fractions.Fraction`, or `decimal.Decimal` instead.

**The route:** first we reproduce the surprise. Then we meet the same kind of problem in ordinary decimal, learn to write fractions in binary (base 2), and find that one tenth never ends in binary. Next we see how a `float` keeps only a fixed number of binary digits, which gives a simple model: *every float is a point on a fixed grid, and every result is rounded to the nearest grid point.* With that model we solve `0.1 + 0.2` completely, measure how the grid spacing changes with size, and finish with safe comparisons and exact alternatives.

## Setup: tools for looking inside numbers

Everything here uses only Python's standard library:

- `math`: comparison helpers, and tools for stepping between neighbouring floats (introduced when we need them).
- `fractions.Fraction`: an exact fraction. `Fraction(3, 10)` and `Fraction("0.3")` both hold exactly three tenths, as a whole-number numerator and denominator, with no rounding. Note the quotes: given *text*, `Fraction` reads the decimal exactly. We will use it as a magnifying glass, because arithmetic with it never rounds.
- `decimal.Decimal`: used in two ways later on. `Decimal(x)` for a float `x` writes the float's exact value out in decimal digits; in the last section it also appears as an alternative number type.

The `assert` line stops with a clear message if your Python is older than 3.9, which lacks `math.ulp` and `math.nextafter`.

In [ ]:
import math
import sys
from decimal import Decimal
from fractions import Fraction

assert sys.version_info >= (3, 9), "This notebook needs Python 3.9+ (math.ulp, math.nextafter)."
print("Python", sys.version.split()[0])

## The surprise, up close

Before explaining anything, let's pin down exactly what goes wrong, so we know what the explanation has to account for.

<!-- la:predict -->
**Predict.** What will the next cell print for:

- (a) `0.1 + 0.2 == 0.3`
- (b) `0.1 + 0.2`
- (c) a running total that starts at `0.0` and has `0.1` added to it ten times, and whether that total `== 1.0`

*My prediction:* 

In [ ]:
print(0.1 + 0.2 == 0.3)
print(0.1 + 0.2)

total = 0.0
for _ in range(10):      # ten items costing 0.1 each
    total += 0.1
print(total, total == 1.0)

<!-- la:interpret -->
**What happened.** `0.1 + 0.2` printed `0.30000000000000004`, a hair *more* than 0.3, so `==` answered `False`. The loop total came out as `0.9999999999999999`, a hair *less* than 1. The errors are tiny (about one part in $10^{16}$, far beyond the digits we typed), but `==` asks for *exact* equality, so tiny is enough to fail.

Two details the explanation must cover: the error is sometimes upwards and sometimes downwards, and it appears even though every number we typed has only one decimal digit.

(We used an explicit loop on purpose. Since Python 3.12 the built-in `sum()` adds floats with extra internal precision, and `sum([0.1] * 10)` happens to give exactly `1.0`. Plain `+` in a loop, which is what most code does, shows the effect.)

<!-- la:misconception -->
> **A tempting mistake:** "Float arithmetic is sloppy; the computer is a bit random with decimals." Run the cell above again: you get exactly the same digits every time. Python's `float` follows a published standard, IEEE 754, which fixes the result of every `+`, `-`, `*` and `/` down to the very last digit, so the same calculation gives the same answer on essentially every machine Python runs on. The error is not noise; it is a precise, repeatable consequence of how `0.1` is stored. That is good news: something precise can be explained and predicted, which is what the rest of this notebook does.

## A warm-up in decimal: when a fraction does not fit

The cause turns out to be a problem you already know from ordinary decimal numbers, so let's meet it there first, where the arithmetic is familiar.

Imagine a calculator that keeps only **4 digits after the decimal point**. Ask it for 1/3. The true value is 0.3333… with 3s forever, so the calculator must cut it to `0.3333`. Now add that three times:

$$0.3333 + 0.3333 + 0.3333 = 0.9999 \neq 1$$

The additions themselves were perfect. The damage happened earlier, when 1/3 was squeezed into four digits. Keep that in mind: **the error is made when the number is stored, before any arithmetic happens.**

Which fractions *do* fit in finitely many decimal digits? Each decimal place is worth a tenth of the one before (tenths, hundredths, thousandths, …), so a finite decimal is always a whole number divided by 10, 100, 1000, …. Since $10 = 2 \times 5$, a fraction in lowest terms ends in decimal exactly when its denominator has no prime factors other than 2 and 5:

- $1/2 = 0.5$, $1/4 = 0.25$, $1/5 = 0.2$, $3/20 = 0.15$: these end.
- $1/3 = 0.333\ldots$, $1/6 = 0.1666\ldots$, $1/7 = 0.142857\ldots$: these repeat forever (3 and 7 are not factors of 10).

We'll call this the **denominator rule**: *a fraction in lowest terms can be written with finitely many digits in a base exactly when every prime factor of its denominator is also a prime factor of the base.*

**Where this warm-up stops matching.** Computers do not use base 10; they use base 2, and that changes which fractions are the awkward ones. In base 10, one tenth is the easy case; in base 2 it is an awkward one. Also, a `float` does not keep a fixed number of digits *after the point* the way this imaginary calculator does; it keeps a fixed number of *significant* digits. The next two sections deal with both differences.

## Binary: place value with twos

To see why `0.1` causes trouble we need to write fractions the way the computer does, in **binary** (base 2). Nothing is new except the place values.

In decimal, each place after the point is worth a tenth of the one before: $0.625 = 6/10 + 2/100 + 5/1000$. In binary each digit is 0 or 1 (a binary digit is called a **bit**), and each place after the point is worth *half* of the one before:

| place after the point | 1st | 2nd | 3rd | 4th | 5th |
|---|---|---|---|---|---|
| worth | 1/2 | 1/4 | 1/8 | 1/16 | 1/32 |

We write a small subscript 2 to mark binary. For example

$$0.101_2 = 1/2 + 0/4 + 1/8 = 5/8 = 0.625$$

(Whole numbers work the same way to the left of the point: $101_2 = 4 + 0 + 1 = 5$.)

Since 2 has only the prime factor 2, the denominator rule now says: **a fraction ends in binary exactly when its denominator, in lowest terms, is a power of two** (2, 4, 8, 16, …). One tenth has the factor 5 in its denominator, so before computing anything the rule already predicts that `0.1` never ends in binary.

<!-- la:worked-example -->
### Worked example: writing 0.375 and 0.1 in binary

**Goal:** find the bits after the point.

**Choosing the method.** In decimal, multiplying 0.375 by 10 gives 3.75: the first decimal digit (3) jumps in front of the point. Multiplying by 2 does the same thing in binary: it shifts every bit one place left, so the bit that jumps in front of the point is the next binary digit. That gives **repeated doubling**:

1. Double the leftover (at the start, the leftover is the number itself).
2. If the result is 1 or more, the next bit is 1: subtract 1 and keep the rest as the new leftover. Otherwise the next bit is 0 and the result is the new leftover.
3. Repeat. Stop if the leftover is 0 (the expansion has ended), or if a leftover comes back that we have seen before (from then on, the bits repeat).

The tempting alternative, guessing which of 1/2, 1/4, 1/8, … add up to the number, works for 0.375 but cannot tell us whether a number like 0.1 *ever* finishes. Doubling can, because it shows the leftovers.

**0.375:**

| step | double the leftover | bit | new leftover |
|---|---|---|---|
| 1 | 2 × 0.375 = 0.75 | 0 | 0.75 |
| 2 | 2 × 0.75 = 1.5 | 1 | 0.5 |
| 3 | 2 × 0.5 = 1.0 | 1 | 0, stop |

So $0.375 = 0.011_2$. **Check:** $1/4 + 1/8 = 3/8 = 0.375$, and the denominator 8 is a power of two, as the rule predicted.

**0.1:**

| step | double the leftover | bit | new leftover |
|---|---|---|---|
| 1 | 2 × 0.1 = 0.2 | 0 | 0.2 |
| 2 | 2 × 0.2 = 0.4 | 0 | 0.4 |
| 3 | 2 × 0.4 = 0.8 | 0 | 0.8 |
| 4 | 2 × 0.8 = 1.6 | 1 | 0.6 |
| 5 | 2 × 0.6 = 1.2 | 1 | 0.2 |

After step 5 the leftover is 0.2, the same leftover we had after step 1. So steps 2–5 now repeat forever, producing the bits 0011 again and again:

$$0.1 = 0.0\,0011\,0011\,0011\ldots_2$$

**Check:** this agrees with the denominator rule (10 contains the factor 5).

*Before reading on: why does a repeated leftover guarantee that the bits repeat?* Each step depends only on the current leftover, so the same leftover must produce the same sequence of steps, and the same bits, all over again.

The next cell does the same procedure in code. It uses `Fraction` so that the doubling and subtracting are exact; doing it with floats would use the very thing we are investigating. Note that we pass the number as text, `Fraction("0.1")`, which means exactly one tenth.

In [ ]:
def binary_digits(x, n=20):
    """Return the first n bits after the point of x (0 <= x < 1), and the leftover.

    x should be a Fraction, so that no rounding happens along the way.
    """
    bits = ""
    for _ in range(n):
        x = 2 * x
        if x >= 1:
            bits += "1"
            x -= 1
        else:
            bits += "0"
        if x == 0:  # nothing left over: the expansion has ended
            break
    return "0." + bits, x


print(binary_digits(Fraction("0.375")))
print(binary_digits(Fraction("0.1"), 24))

The first line shows `0.011` with leftover `0`: finished. The second shows 24 bits of the `0011` pattern and a leftover that is still not zero; asking for more bits would only continue the pattern.

<!-- la:predict -->
**Predict.** Using the denominator rule only (no code), which of `0.5`, `0.625`, `0.3` and `0.7` end in binary, and which repeat forever? For the ones that end, how many bits do they need?

*My prediction:* 

In [ ]:
for text in ["0.5", "0.625", "0.3", "0.7"]:
    bits, leftover = binary_digits(Fraction(text), 24)
    print(f"{text:>5} = {str(Fraction(text)):>5}  ->  {bits:<27} leftover: {leftover}")

<!-- la:interpret -->
**What happened.** `0.5 = 1/2` is $0.1_2$ (one bit) and `0.625 = 5/8` is $0.101_2$ (three bits): their denominators are powers of two, and the leftover reached 0. `0.3 = 3/10` and `0.7 = 7/10` keep a 5 in the denominator; their leftovers never reach 0 and the bits fall into a repeating pattern. In fact, of the one-decimal-place numbers 0.1, 0.2, …, 0.9, only 0.5 ends in binary: every other one, in lowest terms, has a denominator of 10 or 5. Most of the "simple" decimals we type cannot be written exactly in binary.

<!-- la:exercise id=E1 outcomes=1 level=guided -->
**Try it (guided).** Decide whether `0.6875` and `0.2` can be written with finitely many bits.

1. Write each as a fraction in lowest terms (the first line of the next cell does this for you) and use the denominator rule to predict the answer.
2. Check your prediction with `binary_digits`. Remember to pass text to `Fraction`.
3. For the one that ends, check its bits by adding up their place values (1/2, 1/4, 1/8, …).

In [ ]:
# Step 1: Fraction reduces to lowest terms automatically
print(Fraction("0.6875"), Fraction("0.2"))

# Step 2: your binary_digits calls here
...

# Step 3: add up the place values of the bits, e.g. Fraction(1, 2) + Fraction(1, 8)
...

## What a float actually keeps

We now know that `0.1` needs infinitely many bits. A computer has finite memory, so it has to stop somewhere. This section shows where it stops, and therefore what number is really stored when you type `0.1`.

**Significant digits, in decimal first.** Scientists write $6.02 \times 10^{23}$ or $3.0 \times 10^{-5}$: a few *significant digits* (the digits from the first non-zero one onwards) times a power of ten that says where the point goes. With 3 significant digits, 1234.5 becomes $1.23 \times 10^{3}$ and 0.012345 becomes $1.23 \times 10^{-2}$; where the point sits does not cost any digits.

**A float does the same in binary.** A Python `float` stores a number as

$$\pm\ 1.b_1 b_2 \ldots b_{52} \times 2^{e}$$

that is, **53 significant bits** (a leading 1 plus 52 more bits $b_1, \ldots, b_{52}$) times a power of two $2^e$ that places the point, with $-1022 \le e \le 1023$ for ordinary numbers. This format is called IEEE 754 double precision; an optional section near the end shows how it fits in 64 bits.

For 0.1 the bits we found are $0.0001100110011\ldots_2$. Moving the point to just after the first 1 (four places right, so the power is $2^{-4}$) gives

$$0.1 = 1.100110011001\ldots_2 \times 2^{-4}$$

and only 53 significant bits may be kept. What happens to the rest? Python **rounds to the nearest** number that does fit, just as squeezing 1/3 into 4 decimal places gave `0.3333`. For 0.1, the bits that get dropped are worth more than half of the last kept bit, so it rounds *up*: the stored value should be slightly **more** than one tenth. Let's check.

<!-- la:predict -->
**Predict.** `Fraction(x)` for a float `x` (no quotes this time!) shows the float's exact stored value as a fraction, and `Decimal(x)` writes the same value out in full decimal digits.

- Will `Fraction(0.1)` be `1/10`?
- Will `Fraction(0.5)` be `1/2`?
- Is the stored 0.1 above or below one tenth, and by roughly how much?

*My prediction:* 

In [ ]:
print(Fraction(0.1))
print(Decimal(0.1))
print("equal to one tenth?", Fraction(0.1) == Fraction(1, 10))
excess = Fraction(0.1) - Fraction(1, 10)
print("stored 0.1 minus one tenth:", excess, "which is about", float(excess))
print("denominator is 2**55?", Fraction(0.1).denominator == 2**55)
print()
print(Fraction(0.5), Decimal(0.5))

<!-- la:interpret -->
**What happened.** The float written `0.1` is really

$$\frac{3602879701896397}{36028797018963968} = 0.1000000000000000055511151231257827021181583404541015625$$

Its denominator is $2^{55}$, a power of two, as it must be for a number with finitely many bits. It is above one tenth by $1/180143985094819840 \approx 5.55 \times 10^{-18}$, which matches "rounded up". `0.5` needs only one bit, so it is stored exactly: `Fraction(0.5)` is exactly `1/2`.

So the computer never sees one tenth. Every time you type `0.1`, you get this nearby binary fraction instead.

<!-- la:misconception -->
> **A tempting mistake:** "`print(0.1)` shows `0.1`, so the float holds exactly 0.1." When Python displays a float, it shows the **shortest** decimal text that would turn back into the *same* float. The text `0.1` is much shorter than the 55-digit truth and still converts back to the same stored value, so that is what you see: a label, not the value. The evidence was already in front of us: `0.1 + 0.2` displayed as `0.30000000000000004` because no shorter text converts back to that particular float. To see what is really stored, use `Fraction(x)` or `Decimal(x)`.

## The grid model: round, compute, round again

We now have two facts: typed decimals are rounded when they become floats, and floats hold only 53 significant bits. Together they give one small model that explains every result so far, and lets us solve `0.1 + 0.2` completely.

**The model.**

1. The numbers a float can hold form a fixed set of points on the number line. Call it **the grid**.
2. Turning text like `0.1` into a float picks the **nearest grid point**.
3. Each `+`, `-`, `*`, `/` takes the two stored inputs, works out the **exact** answer, and then rounds it to the nearest grid point. This is the IEEE 754 guarantee mentioned earlier; the standard calls it *correct rounding*.
4. **Ties:** if the exact answer is exactly halfway between two grid points, the result is the one whose last significant bit is 0. This is called *round half to even* ("even" because a binary number ending in 0 is even). Decimal has the same rule when you want it: rounding 0.125 to two places by round-half-to-even gives 0.12, because 2 is even.

**Spacing near a number.** For any $x$ between $2^e$ and $2^{e+1}$, the floats there all look like $1.b_1\ldots b_{52} \times 2^e$, so neighbouring floats differ by one unit in the 52nd bit after the point: a gap of $2^{e-52}$. This gap is called an **ulp**, short for *unit in the last place*. For example, 0.3 lies between $1/4 = 2^{-2}$ and $1/2 = 2^{-1}$, so near 0.3 the grid points are $2^{-2-52} = 2^{-54} \approx 5.55 \times 10^{-17}$ apart.

Python has two tools for walking the grid: `math.nextafter(x, y)` returns the next float after `x` in the direction of `y`, and `math.ulp(x)` returns the gap from `x` up to the next float.

**The ruler picture, and where it stops working.** You can picture the floats as tick marks on a ruler, with every result snapping to the nearest tick. Unlike a real ruler, though, the ticks are not evenly spaced: the spacing doubles every time you pass a power of two (we measure that in the next section). The ruler also ends: beyond about $1.8 \times 10^{308}$ there are no more ticks, which is outside this lesson.

<!-- la:worked-example -->
### Worked example: following `0.1 + 0.2` to the last bit

**Goal:** explain exactly why `0.1 + 0.2` gives `0.30000000000000004`, while `0.3` gives a different float.

**Choosing the method.** The model says several roundings are involved: storing 0.1, storing 0.2, rounding the sum, and (for the comparison) storing 0.3. To see each one we need arithmetic that does *not* round, so every comparison and difference below uses `Fraction`. Measuring the error with floats, say `0.1 + 0.2 - 0.3`, would itself be rounded, so it could not show us where the rounding happened.

**Step 1: what is actually added?** *Goal:* find the stored inputs. Compare each stored value with the decimal it came from.

In [ ]:
for text in ["0.1", "0.2", "0.3"]:
    stored = Fraction(float(text))   # what the float holds
    intended = Fraction(text)        # the exact decimal we typed
    direction = "above" if stored > intended else "below"
    print(f"stored {text} = {Decimal(float(text))}  ({direction} {text})")

So 0.1 and 0.2 were both rounded *up* when stored, and 0.3 was rounded *down*.

<!-- la:predict -->
**Step 2: the exact sum.** *Goal:* see what `+` is asked to round. Before running: is the exact sum of the stored 0.1 and the stored 0.2 above or below 3/10? Above or below the *stored* 0.3?

*My prediction:* 

In [ ]:
a, b, c = Fraction(0.1), Fraction(0.2), Fraction(0.3)
exact_sum = a + b          # Fraction arithmetic: no rounding at all
print("exact sum of stored inputs:", exact_sum)
print("above 3/10?        ", exact_sum > Fraction(3, 10))
print("above stored 0.3?  ", exact_sum > c)

<!-- la:interpret -->
The exact sum is above 3/10 (two upward errors added together) and therefore also above the stored 0.3, which is below 3/10. The exact sum is *not itself a grid point*, so rule 3 of the model applies: it must be rounded to one.

**Step 3: which grid points are nearby?** *Goal:* find the two candidates. The stored 0.3 is a grid point just below the exact sum, and the next grid point up is `math.nextafter(0.3, 1.0)`. We measure the distance from the exact sum to each, in units of the grid spacing near 0.3, which is $2^{-54}$ (from the spacing formula above). *Check:* `math.ulp(0.3)` should equal $2^{-54}$.

In [ ]:
lower = 0.3
upper = math.nextafter(0.3, 1.0)
step = Fraction(math.ulp(0.3))
print("grid spacing near 0.3 is 2**-54:", step == Fraction(1, 2**54))
print("lower < exact sum < upper:      ", Fraction(lower) < exact_sum < Fraction(upper))
print("distance to lower, in grid steps:", (exact_sum - Fraction(lower)) / step)
print("distance to upper, in grid steps:", (Fraction(upper) - exact_sum) / step)

**Step 4: round.** *Goal:* decide which candidate wins. The exact sum is **exactly halfway** between the two grid points, half a step from each. That is the tie case, so rule 4 applies: the winner is the neighbour whose last significant bit is 0. Both neighbours lie between $2^{-2}$ and $2^{-1}$, so each has the form $1.b_1 \ldots b_{52} \times 2^{-2}$. Moving the point 52 places to the right turns $1.b_1 \ldots b_{52}$ into a 53-bit whole number and costs a factor $2^{-52}$, so each neighbour is (a 53-bit whole number) × $2^{-54}$, ending in the bit $b_{52}$. Multiplying by $2^{54}$ recovers that whole number, and its last bit is 0 exactly when it is even.

In [ ]:
for name, x in [("lower (stored 0.3)", lower), ("upper", upper)]:
    m = Fraction(x) * 2**54
    print(f"{name:>18}: {m}  ->  {'even' if m % 2 == 0 else 'odd'}")
print()
print("0.1 + 0.2 lands on upper:", 0.1 + 0.2 == upper, "->", 0.1 + 0.2)
print("0.1 + 0.2 - 0.3 is one grid step:", 0.1 + 0.2 - 0.3 == 2**-54, "->", 0.1 + 0.2 - 0.3)

**Step 5: conclude and check.** The tie went to the even neighbour, `upper`, which displays as `0.30000000000000004`. The float written `0.3` is `lower`. They are neighbouring grid points, one step ($2^{-54} \approx 5.55\times10^{-17}$) apart, so `0.1 + 0.2 == 0.3` compares two different points and answers `False`. **Check:** the last line confirms that `0.1 + 0.2 - 0.3` is exactly one grid step, $2^{-54}$.

Every part of the model was needed: rounding on the way in pushed 0.1 and 0.2 up and 0.3 down; correct rounding after the addition put the sum on a grid point; and the tie rule decided which one.

<!-- la:exercise id=E2 outcomes=2 level=guided -->
**Try it (guided).** In Python, `0.1 + 0.7` gives `0.7999999999999999`, not `0.8`. Explain why by following the worked example:

1. For 0.1, 0.7 and 0.8, is each stored value above or below the decimal it came from?
2. Compute the exact sum of the stored 0.1 and the stored 0.7 with `Fraction`. Is it above or below the stored 0.8?
3. Find the grid points either side of the exact sum (start from `r = 0.1 + 0.7` and use `math.nextafter`), and measure the distances in grid steps using `math.ulp(0.8)`. Which is nearer? Is there a tie this time?
4. In one or two sentences: where did the error come from?

Then do a quick check on `1.1 + 2.2` versus `3.3`: compare `Fraction(1.1) + Fraction(2.2)` with `Fraction(1.1 + 2.2)`. Did the addition itself have to round? What does that tell you about where *this* error came from?

In [ ]:
# 1. stored values vs the decimals they came from
for text in ["0.1", "0.7", "0.8"]:
    ...

# 2. exact sum of the stored inputs
exact = ...

# 3. neighbouring grid points and distances, in grid steps
...

# 1.1 + 2.2 check
...

## The grid gets coarser as numbers grow

So far we zoomed in near 0.3, where neighbouring floats are about $5.55 \times 10^{-17}$ apart. To compare floats safely we need to know how far apart neighbours are *in general*, and the spacing formula $2^{e-52}$ says that depends on the size of the number.

**Why it grows.** 53 significant bits is like a calculator with a fixed number of *significant* digits. With 4 significant decimal digits, numbers near 1 can be `1.000, 1.001, 1.002, …` (gap 0.001), but numbers near 1000 can only be `1000, 1001, 1002, …` (gap 1). The gap is always about the same *fraction* of the number. Floats behave the same way in binary.

<!-- la:predict -->
**Predict.** The next cell prints `math.ulp(x)` (the gap from `x` to the next float) and `math.ulp(x) / x` (that gap as a fraction of `x`) for x = 0.3, 1, 1000, one million and $10^{17}$. Roughly how big is the gap at $10^{17}$: tiny, about 1, or bigger than 1? And what do you expect in the gap ÷ x column?

*My prediction:* 

In [ ]:
for x in [0.3, 1.0, 1000.0, 1e6, 1e17]:
    print(f"x = {x:<8g} gap to next float = {math.ulp(x):<10.3g} gap / x = {math.ulp(x) / x:.3g}")

<!-- la:interpret -->
**What happened.** The absolute gap changes enormously, from about $5.55 \times 10^{-17}$ near 0.3 to **16** near $10^{17}$: up there, floats are 16 apart, so most whole numbers do not exist as floats at all. But the relative gap (gap ÷ x) stays between about $1.1 \times 10^{-16}$ and $2.2 \times 10^{-16}$ everywhere. Rounding to the nearest grid point moves a value by at most half a gap, so **each rounding changes a number by at most about one part in $9 \times 10^{15}$** (that is, by at most $2^{-53} \approx 1.1 \times 10^{-16}$ of its size). In decimal terms, a float carries roughly 15 to 17 significant digits, wherever the point is, which is why the errors we saw were only about one part in $10^{16}$.

<!-- la:misconception -->
> **A tempting mistake:** "Floats are accurate to about 16 decimal places." Places *after the point* would mean a fixed absolute accuracy, and the table shows there is no such thing: near $10^{17}$ the nearest float can be up to 8 away. The next cell converts the whole number 123456789012345678 to a float and back; the nearest float is 2 away, so you get 123456789012345680. The correct model: floats keep about 16 *significant* digits (a relative accuracy), wherever the decimal point falls.

In [ ]:
print(int(float(123456789012345678)))

<!-- la:exercise id=E3 outcomes=3 level=independent -->
**Exercise 3.** (a) Predict, then check: is `1e17 + 1 == 1e17` true or false? What about `1e17 + 9 == 1e17`? Explain both results using `math.ulp` and the grid model.

(b) Find the smallest whole number `k` for which `float(2**k) + 1.0 == float(2**k)` is `True`, and explain why that particular `k` is connected to the 53 significant bits.

In [ ]:
# your code here
...

## Comparing floats safely

We can now explain every surprise so far. What remains is the practical question: when your program needs to ask "are these two numbers equal?", what should it write instead of `==`?

`==` asks whether two floats are **the same grid point**. That is the right question when the values were computed exactly: small whole numbers, halves, quarters and other short binary fractions. But after rounding, two routes to "the same" number can end on neighbouring grid points, as `0.1 + 0.2` and `0.3` did. So instead we ask whether the two numbers are **close enough**, which needs a *tolerance*, the largest difference we are willing to ignore. There are two kinds:

- **Absolute tolerance**: a fixed amount, as in "within half a millimetre". Test: `abs(a - b) <= abs_tol`.
- **Relative tolerance**: a fraction of the numbers' size, as in "within one part in a billion". Test: `abs(a - b) <= rel_tol * max(abs(a), abs(b))`. This matches the grid, whose spacing is relative.

`math.isclose(a, b, rel_tol=1e-09, abs_tol=0.0)` says "close" if *either* test passes. In the words of the Python documentation, it computes

```python
abs(a - b) <= max(rel_tol * max(abs(a), abs(b)), abs_tol)
```

The defaults are a relative tolerance of $10^{-9}$ and **no** absolute tolerance.

<!-- la:predict -->
**Predict.** True or false, for each line?

- (1) `abs((0.1 + 0.2) - 0.3) < 1e-9`: a fixed tolerance, small numbers
- (2) `abs((1e17 + 16) - 1e17) < 1e-9`: a fixed tolerance, neighbouring floats near $10^{17}$ (`1e17 + 16` is the very next float)
- (3) `abs(1e-12 - 2e-12) < 1e-9`: a fixed tolerance, two tiny numbers, one twice the other
- (4) `math.isclose(0.1 + 0.2, 0.3)`
- (5) `math.isclose(1e17 + 16, 1e17)`
- (6) `math.isclose(1e-12, 2e-12)`
- (7) `math.isclose(0.1 + 0.2 - 0.3, 0.0)`: is the leftover "zero"?

*My prediction:* 

In [ ]:
print("(1)", abs((0.1 + 0.2) - 0.3) < 1e-9)
print("(2)", abs((1e17 + 16) - 1e17) < 1e-9)
print("(3)", abs(1e-12 - 2e-12) < 1e-9)
print("(4)", math.isclose(0.1 + 0.2, 0.3))
print("(5)", math.isclose(1e17 + 16, 1e17))
print("(6)", math.isclose(1e-12, 2e-12))
print("(7)", math.isclose(0.1 + 0.2 - 0.3, 0.0))

<!-- la:interpret -->
**What happened.** The fixed tolerance `1e-9` gave True, **False**, **True** for lines 1 to 3. Line 2 is too strict: near $10^{17}$ the closest two different floats can be is 16 apart, so no tiny fixed tolerance can ever call them equal. Line 3 is too loose: $10^{-12}$ and $2 \times 10^{-12}$ differ by a factor of two, yet both count as "within 1e-9" simply because both are tiny. A fixed absolute tolerance only makes sense once you know the size of your numbers.

`math.isclose`, with its relative default, got lines 4 to 6 right (True, True, False), because its tolerance scales with the numbers, just as the grid does. Line 7 is the surprise: `False`.

<!-- la:misconception -->
> **A tempting mistake:** "`math.isclose` with the defaults handles everything, including checking that something is zero." When one side is 0, the relative test becomes `abs(x) <= 1e-9 * abs(x)`, which is false for every nonzero `x`, however small (the Python documentation warns about exactly this). The leftover `0.1 + 0.2 - 0.3` is about $5.55 \times 10^{-17}$, not zero, so the answer is `False`. **The fix:** whenever one side can be zero or near it, supply an `abs_tol`, chosen from the size of the numbers that went *into* the calculation.
>
> And the older habit, "always use `abs(a - b) < 1e-9`", fails in both directions, as lines 2 and 3 showed.

<!-- la:worked-example -->
### Worked example: choosing tolerances for the running example

**Problem.** Our loop added `0.1` ten times. We want two checks: (i) "the total is 1", and (ii) "the leftover `0.1 + 0.2 - 0.3` is zero".

**Step 1: how big an error should we expect?** *Goal:* know what the tolerance has to absorb. Each rounding changes a value by at most about $1.1 \times 10^{-16}$ of its size (previous section). Ten additions with totals no bigger than 1 make ten roundings, so the total can be off by at most about $10^{-15}$.

**Step 2: how big a difference would matter?** *Goal:* know what the tolerance must *not* hide. In a typical program that reports results to a few significant digits, a difference of one part in a million would certainly matter.

**Step 3: check (i) with a relative tolerance.** *Why this kind:* the numbers are around 1, not near zero, so a fraction of their size is the natural yardstick. The default `rel_tol=1e-9` is a million times larger than the expected error ($10^{-15}$) and a thousand times smaller than the difference we care about, so we keep the default. *Why not `round()` both sides instead?* Rounding to a fixed number of places has cliff edges that can split neighbouring floats: `0.125` and the very next float, `0.12500000000000003`, round to two places as `0.12` and `0.13`. A tolerance has no such edges.

**Step 4: check (ii) with an absolute tolerance.** *Why this kind:* one side is zero, so the relative test cannot help (see the mistake above). The inputs were about 0.3 in size, where rounding errors are around $10^{-16}$. An `abs_tol` of $10^{-12}$ is ten thousand times bigger than that noise, yet still tiny compared with the inputs. *Why base it on the inputs, not the result?* The leftover's size is exactly what we are trying to judge, so it cannot set its own tolerance; the noise it contains comes from rounding the inputs.

**Step 5: check the checks.** A tolerance that says "yes" to everything is useless, so confirm that each check still rejects a value that is genuinely different.

In [ ]:
total = 0.0
for _ in range(10):
    total += 0.1
leftover = 0.1 + 0.2 - 0.3

# (i) the total is 1: relative tolerance (default rel_tol=1e-9)
print("total is 1:           ", math.isclose(total, 1.0))
# (ii) the leftover is 0: absolute tolerance chosen from the inputs' size
print("leftover is zero:     ", math.isclose(leftover, 0.0, abs_tol=1e-12))

# Step 5: both checks still reject genuinely different values
print("1.000001 counts as 1? ", math.isclose(1.000001, 1.0))
print("1e-6 counts as zero?  ", math.isclose(1e-6, 0.0, abs_tol=1e-12))

<!-- la:exercise id=E4 outcomes=4 level=independent -->
**Exercise 4.** A program tracks positions along a 10 km track, in metres (values from 0 to 10000). Positions are measured to the millimetre, and two positions should count as the same if they differ by at most half a millimetre. Write `positions_match(a, b)` using `math.isclose`, justify the tolerances you chose, and test it on these pairs (expected answers in brackets):

- `0.1 + 0.2` and `0.3` (same)
- `0.0` and `0.0004` (same)
- `0.0` and `0.0006` (different)
- `5000.1 + 0.2` and `5000.3` (same)
- `9999.999` and `10000.0` (different)

In [ ]:
def positions_match(a, b):
    ...  # your code here


# your tests here
...

## When "close enough" is not enough: integers, Fraction and Decimal

Tolerances are the right tool when your numbers are measurements or approximations anyway. Sometimes, though, close is not acceptable. Money is the classic case: a bill must add up to the cent, and an auditor will compare totals with `==`. Then the fix is to change *what you store*, not how you compare. Python offers three alternatives to `float`:

- **Whole numbers (`int`).** Python integers are exact at any size. Store money as whole **cents**: `1999 + 470 + 70 == 2539` is exactly true. Simplest and fastest.
- **`Fraction`.** Exact for any fraction, including 1/3. Slower, and the numerators and denominators can grow large in long calculations.
- **`Decimal`.** Does its arithmetic in base 10, so `0.1` can be exact *if you build it from text*: `Decimal("0.1")`. It keeps a fixed number of significant decimal digits (28 by default) and offers the rounding rules accountants use.

<!-- la:predict -->
**Predict.** True or false, for each line?

- (1) `Decimal("0.1") + Decimal("0.2") == Decimal("0.3")`
- (2) `Decimal(0.1) + Decimal(0.2) == Decimal("0.3")` (note: no quotes on the left)
- (3) `Decimal(1) / Decimal(3) * 3 == 1`
- (4) `Fraction(1, 10) + Fraction(2, 10) == Fraction(3, 10)`
- (5) `10 + 20 == 30`: the same sum in whole cents

*My prediction:* 

In [ ]:
print("(1)", Decimal("0.1") + Decimal("0.2") == Decimal("0.3"))
print("(2)", Decimal(0.1) + Decimal(0.2) == Decimal("0.3"))
print("(3)", Decimal(1) / Decimal(3) * 3 == 1, "->", Decimal(1) / Decimal(3) * 3)
print("(4)", Fraction(1, 10) + Fraction(2, 10) == Fraction(3, 10))
print("(5)", 10 + 20 == 30)

<!-- la:interpret -->
**What happened.** Lines 1, 4 and 5 are `True`: decimals built from text, fractions and whole cents are all exact for these values. Line 2 is `False`: `Decimal(0.1)` receives the *float* 0.1, which was already rounded to the binary fraction we found earlier, and `Decimal` faithfully keeps all 55 digits of it. The error came in before `Decimal` ever saw the number. Line 3 is `False` too: `Decimal` works in base 10, so by the denominator rule 1/3 cannot end; it is rounded to 28 digits, `0.3333333333333333333333333333`, and three of those make `0.9999999999999999999999999999`.

<!-- la:misconception -->
> **A tempting mistake:** "Switching to `Decimal` makes my numbers exact." It changes the *base* from 2 to 10, which makes typed decimals like 0.1 exact, but only if they arrive as text (`Decimal("0.1")`), not through a float (`Decimal(0.1)`). And it still keeps finitely many digits, so fractions like 1/3 are still rounded. `Decimal` removes the *binary* surprise, not every rounding.

**Which one to use.**

| Your numbers are… | Use |
|---|---|
| measurements, scientific values, graphics, statistics: already approximate | `float`, compared with `math.isclose` |
| money, or counts of a fixed smallest unit | `int` in the smallest unit (such as cents), or `Decimal` built from text |
| exact ratios such as 1/3, in a small calculation | `Fraction` |

<!-- la:exercise id=E5 outcomes=5 level=transfer -->
**Exercise 5.** The situation changes: exactness is now required. A shop's till reads prices **as text** from a price list, and an auditing script compares receipt totals with `==`. A customer buys items priced `"19.99"`, `"4.70"` and `"0.70"`; the correct total is 25.39.

1. Show what goes wrong if the till converts each price with `float()` and adds them in a loop.
2. Write `till_total(prices)`, which takes a list of price strings and returns an exact total, and show that `==` against the expected total is now safe.
3. In a sentence each: why is `math.isclose` not the right fix here, and what would go wrong if the till used `Decimal(float(price))`?

In [ ]:
prices = ["19.99", "4.70", "0.70"]

# 1. the float version
...

# 2. an exact version
def till_total(prices):
    ...  # your code here


...

<!-- la:optional -->
<details>
<summary>Optional: how a float fits in 64 bits, and why <code>sum()</code> behaved differently</summary>

Nothing later in the notebook depends on this.

**The 64 bits.** A float occupies 64 bits: 1 sign bit, 11 bits for the power $e$, and 52 bits for $b_1 \ldots b_{52}$. The leading `1.` is not stored, because every ordinary float starts with it. (Very tiny numbers, below about $2.2 \times 10^{-308}$, are stored slightly differently; they are outside this lesson.)

`float.hex()` shows the stored bits in hexadecimal, where each hex digit stands for 4 bits (`9` = `1001`, `a` = `1010`, `3` = `0011`, `4` = `0100`):

- `(0.1).hex()` is `'0x1.999999999999ap-4'`: that is $1.1001\,1001 \ldots 1001\,1010_2 \times 2^{-4}$. The repeating `1001` groups are 0.1's pattern; the final `a` (`1010`) is the last `1001` rounded *up*.
- `(0.3).hex()` is `'0x1.3333333333333p-2'` and `(0.1 + 0.2).hex()` is `'0x1.3333333333334p-2'`: the two floats from the worked example differ only in the last hex digit, one grid step, and the result's final `4` (`0100`) ends in the bit 0, as the tie rule required.

**About `sum()`.** Since Python 3.12, `sum()` of floats keeps extra precision internally while adding, which is why `sum([0.1] * 10) == 1.0` there, while the plain loop gives `0.9999999999999999`. `math.fsum()` goes further and rounds only once, at the end. Both reduce *accumulated* error; neither changes how `0.1` itself is stored, so `math.fsum([0.1, 0.2]) == 0.3` is still `False`.

</details>

## Check your understanding

<!-- la:exercise id=E6 outcomes=2 level=check -->
**Exercise 6.** Without code, explain to a friend who knows Python but has not read this notebook why `0.1 + 0.2 != 0.3`. Use at most six sentences, and mention binary, rounding when a number is stored, and rounding after the addition.

*Your explanation:* 

<!-- la:synthesis -->
## So why is `0.1 + 0.2 != 0.3`?

We started with Python insisting that `0.1 + 0.2 == 0.3` is `False`. Here is the full answer, built from the pieces of this notebook.

A float stores a binary fraction with 53 significant bits. By the denominator rule, one tenth (denominator $10 = 2 \times 5$) has no finite binary form; it is $0.0\,0011\,0011\ldots_2$ forever. So typing `0.1` gives the nearest grid point instead, a value slightly above one tenth. The same happens to `0.2` (also slightly above) and `0.3` (slightly below). Then `+` adds the two stored values exactly and rounds to the grid: the exact sum lands precisely halfway between the stored `0.3` and the next float up, and round-half-to-even sends it up. The result, `0.30000000000000004`, is one grid step ($2^{-54}$) away from the float written `0.3`, and `==` compares grid points exactly. Nothing is random or buggy; it is the decimal calculator's problem with 1/3, moved to base 2.

Our running loop works the same way: each of the ten `+= 0.1` steps rounds, and the total ends one grid step below `1.0`, at `0.9999999999999999`. The safe check is `math.isclose(total, 1.0)`, and if the total were money, the right fix would be to store whole cents or `Decimal` values built from text.

**What you can now do:**

- decide from a fraction's denominator whether a decimal number is stored exactly, and work out its binary digits;
- reveal any float's true value with `Fraction(x)` or `Decimal(x)`, and trace a calculation through its roundings;
- measure the spacing between floats with `math.ulp`, and expect about 16 significant digits, not 16 decimal places;
- compare floats with `math.isclose`, using a relative tolerance for ordinary values and an absolute tolerance near zero;
- switch to whole cents, `Decimal` built from text, or `Fraction` when exact answers are required.

**What this notebook did not cover:** infinities and `nan`, overflow, very tiny numbers, and how errors grow over long calculations, in particular *cancellation*, where subtracting two nearly equal numbers leaves mostly rounding error.

**A sensible next step:** read the Python tutorial chapter "Floating-Point Arithmetic: Issues and Limitations" (https://docs.python.org/3/tutorial/floatingpoint.html), which re-derives the 0.1 example in its own way and introduces `math.fsum`; then look up *catastrophic cancellation*.

<!-- la:solutions -->
## Hints and solutions

Try each exercise before reading on. Hints come first and give away as little as possible; full solutions follow.

### Hints

<!-- la:hint id=E1 -->
**E1, hint 1.** Look at the two denominators that `Fraction` printed. Which one is a power of two?

**E1, hint 2.** Call `binary_digits(Fraction("0.6875"))` and `binary_digits(Fraction("0.2"), 24)`. A 1 bit in place k after the point is worth $1/2^k$.

<!-- la:hint id=E2 -->
**E2, hint 1.** Step 1 is the loop from Step 1 of the worked example, with different numbers.

**E2, hint 2.** The grid points around the exact sum are `r = 0.1 + 0.7` and `math.nextafter(r, 1.0)`. Divide each distance by `Fraction(math.ulp(0.8))`.

<!-- la:hint id=E3 -->
**E3, hint 1.** `math.ulp(1e17)` gives the grid spacing there. How far from `1e17` must the exact answer be before the nearest grid point is a different one?

**E3, hint 2.** For (b), loop `k` over `range(40, 60)` and stop at the first `True`. Then look at `math.ulp(float(2**k))` for that `k` and for `k - 1`, and ask where exactly $2^k + 1$ sits between grid points.

<!-- la:hint id=E4 -->
**E4, hint 1.** Is the requirement stated as a fixed amount, or as a fraction of the position?

**E4, hint 2.** Half a millimetre is 0.0005 m. What does the relative tolerance need to be so that it does not interfere?

<!-- la:hint id=E5 -->
**E5, hint 1.** The prices arrive as text. Use the text directly; never let a price become a float.

**E5, hint 2.** `sum()` works on `Decimal` values. For whole cents, `Decimal(price) * 100` turns a two-decimal price into an exact whole number of cents.

<!-- la:hint id=E6 -->
**E6, hint.** Go in order: what a float can hold, what happens to 0.1 (and 0.2, and 0.3) when stored, what `+` does with the stored values, and what `==` compares.

### Solutions

<!-- la:solution id=E1 -->
**Solution to E1.** `Fraction("0.6875")` is `11/16`; its denominator $16 = 2^4$ is a power of two, so it ends, using at most 4 bits. `Fraction("0.2")` is `1/5`; the factor 5 means it repeats forever.

```python
print(binary_digits(Fraction("0.6875")))      # ('0.1011', Fraction(0, 1))
print(binary_digits(Fraction("0.2"), 24))     # ('0.001100110011001100110011', Fraction(1, 5))
print(Fraction(1, 2) + Fraction(1, 8) + Fraction(1, 16))   # 11/16
```

So $0.6875 = 0.1011_2$, and the check $1/2 + 1/8 + 1/16 = 11/16 = 0.6875$ confirms it. For 0.2 the leftover is never 0 and the bits repeat `0011` (0.2 is twice 0.1, so its bits are 0.1's shifted one place left).

*Most likely wrong answer:* "0.2 is a short decimal, so it is exact." Being short in decimal says nothing about binary. Another trap: writing `Fraction(0.2)` *without* quotes. That expansion does end, after 54 bits, because it expands the already-rounded float rather than one fifth.

<!-- la:solution id=E2 -->
**Solution to E2.**

```python
for text in ["0.1", "0.7", "0.8"]:
    stored, intended = Fraction(float(text)), Fraction(text)
    print(text, "above" if stored > intended else "below")   # above, below, above

exact = Fraction(0.1) + Fraction(0.7)
print(exact < Fraction(0.8))                  # True: already below the stored 0.8

r = 0.1 + 0.7
up = math.nextafter(r, 1.0)
step = Fraction(math.ulp(0.8))
print((exact - Fraction(r)) / step)           # 1/4
print((Fraction(up) - exact) / step)          # 3/4
print(up == 0.8)                              # True: the upper neighbour is the stored 0.8
```

0.1 was stored slightly too big (by about $5.6 \times 10^{-18}$), but 0.7 was stored too small by more (about $4.4 \times 10^{-17}$), so the exact sum of the stored inputs is below 8/10, while the stored 0.8 is above 8/10. The exact sum sits a quarter of a step above `0.7999999999999999` and three quarters of a step below the stored 0.8. That is no tie: the nearer grid point is the lower one. The error comes from the rounding of the inputs on the way in; the addition's own rounding only picked the nearest grid point, which happened not to be the stored 0.8.

`1.1 + 2.2`: `Fraction(1.1) + Fraction(2.2) == Fraction(1.1 + 2.2)` is `True`, so the exact sum of the stored inputs was already a grid point and the addition did not round at all. Yet `1.1 + 2.2` is `3.3000000000000003`, not `3.3`, because 1.1 and 2.2 were stored slightly too big and 3.3 slightly too small. Here the error comes *entirely* from storing the decimals.

*Most likely wrong answer:* "the addition was inaccurate". The `1.1 + 2.2` case shows the addition can be perfect and the comparison still fail.

<!-- la:solution id=E3 -->
**Solution to E3.**

(a) `math.ulp(1e17)` is `16.0`, so the grid points around $10^{17}$ are $10^{17}$ and $10^{17} + 16$. The exact value $10^{17} + 1$ is 1 away from the first and 15 from the second, so it rounds back to `1e17`: `1e17 + 1 == 1e17` is `True`. The exact value $10^{17} + 9$ is 9 from the first and 7 from the second, so it rounds *up* to $10^{17} + 16$: the comparison is `False`, and `1e17 + 9` displays as `1.0000000000000002e+17`.

(b)

```python
for k in range(40, 60):
    if float(2**k) + 1.0 == float(2**k):
        print(k)                               # 53
        break
print(math.ulp(float(2**52)), math.ulp(float(2**53)))   # 1.0 2.0
```

For numbers from $2^{52}$ up to $2^{53}$ the spacing is $2^{52-52} = 1$, so every whole number there is a float and $2^{52} + 1$ is stored exactly (`False` at k = 52, and at every smaller k the spacing is even finer). From $2^{53}$ on, the spacing is 2. Then $2^{53} + 1$ is exactly halfway between $2^{53}$ and $2^{53} + 2$: a tie. $2^{53} = 1.000\ldots0_2 \times 2^{53}$ ends in the bit 0, so round-half-to-even keeps $2^{53}$, and the answer is `True`. The link to 53 significant bits: a whole number up to $2^{53}$ needs at most 53 bits, but $2^{53} + 1$ would need 54, one more than a float has.

*Most likely wrong answer:* k = 54, from assuming the tie at $2^{53} + 1$ rounds up.

<!-- la:solution id=E4 -->
**Solution to E4.**

```python
def positions_match(a, b):
    return math.isclose(a, b, rel_tol=0.0, abs_tol=0.0005)

pairs = [(0.1 + 0.2, 0.3, True), (0.0, 0.0004, True), (0.0, 0.0006, False),
         (5000.1 + 0.2, 5000.3, True), (9999.999, 10000.0, False)]
for a, b, expected in pairs:
    print(a, b, positions_match(a, b), positions_match(a, b) == expected)   # all True in the last column
```

*Why these tolerances.* The requirement is stated as a fixed amount, half a millimetre wherever you are on the track, so this is an absolute tolerance: `abs_tol=0.0005` metres. Setting `rel_tol=0.0` makes the rule say exactly what the specification says. (Leaving the default `1e-9` would also work here, since at 10000 m it allows only 0.00001 m, far less than 0.0005, but it adds nothing.) Rounding noise is not a worry: even at 10000 m, `math.ulp(10000.0)` is about $1.8 \times 10^{-12}$ m, far below half a millimetre.

*Most likely wrong answers:* the default `math.isclose(a, b)`, which says `False` for `0.0` and `0.0004` (the comparison-with-zero trap); or `rel_tol=0.0005`, which at 10 km would accept differences of 5 m.

<!-- la:solution id=E5 -->
**Solution to E5.**

```python
prices = ["19.99", "4.70", "0.70"]

total = 0.0
for p in prices:
    total += float(p)
print(total, total == 25.39)                       # 25.389999999999997 False

def till_total(prices):
    return sum(Decimal(p) for p in prices)         # built from text: exact

print(till_total(prices), till_total(prices) == Decimal("25.39"))   # 25.39 True

def till_total_cents(prices):                      # alternative: whole cents
    return sum(int(Decimal(p) * 100) for p in prices)

print(till_total_cents(prices) == 2539)            # True
```

*Why not `math.isclose`?* The governing condition changed: the rule is now exact equality, checked by someone else's `==`. A tolerance would only hide the error inside our own check; the stored total would still not be 25.39, and the auditor's comparison would still fail. The tolerance method no longer fits, so we change the representation instead.

*Why not `Decimal(float(price))`?* `float(price)` has already rounded to binary; `Decimal` then keeps that rounded value exactly (for example `Decimal(19.99)` is `19.989999999999998436805981327779591083526611328125`), so the total, `25.38999999999999857003274428`, is still not `Decimal("25.39")`.

<!-- la:solution id=E6 -->
**Model answer to E6.** "Python stores a float in binary with 53 significant bits. One tenth cannot be written exactly in binary, because its denominator contains a 5, so `0.1` is stored as the nearest binary fraction, which is slightly too big; `0.2` is also stored slightly too big and `0.3` slightly too small. Adding works out the exact sum of the two stored values and rounds it to the nearest float; here the exact sum was exactly halfway between two floats, and the tie rule picked the upper one, `0.30000000000000004`. That is one step above the float stored for `0.3`, and `==` checks for exactly the same float, so it says `False`."

*A weak answer* says only "computers are imprecise with decimals". It does not say where the error enters (storage, then rounding after the addition), so it cannot predict that `0.5 + 0.25 == 0.75` is `True`.